# 03 — TON_IoT ETL Pipeline

**Abstract:** This notebook ingests the raw TON_IoT `train_test_network.csv` dataset into the SentinelMesh PostgreSQL staging layer. It maps vendor-specific attack-type labels to the unified SentinelMesh taxonomy (`attack_family`, `unified_label`, `is_attack`), normalises selected network-traffic columns, and bulk-inserts the harmonised records into `stg_harmonized` in 5,000-row chunks. A final verification query confirms row counts and label distributions for both TON_IoT and CICIoT2023 sources.

**Pipeline position:** Runs after `01_data_exploration.ipynb` and before `04_prediction_writeback.ipynb`.

---

## Stage 1 — Data Loading & Initial Exploration


In [1]:
import pandas as pd
import os

TON_PATH = os.path.expanduser("~/sentinelmesh/data/TON_IoT/Train_Test_datasets/Train_Test_Network_dataset/train_test_network.csv")

df_ton = pd.read_csv(TON_PATH, low_memory=False)

print(f"Shape        : {df_ton.shape}")
print(f"Columns      : {list(df_ton.columns)}")
print(f"\nLabel counts :\n{df_ton['label'].value_counts()}")
print(f"\nType counts  :\n{df_ton['type'].value_counts()}")
print(f"\nProto counts :\n{df_ton['proto'].value_counts().head(10)}")
print(f"\nNull counts  :\n{df_ton.isnull().sum()[df_ton.isnull().sum() > 0]}")

Shape        : (211043, 44)
Columns      : ['src_ip', 'src_port', 'dst_ip', 'dst_port', 'proto', 'service', 'duration', 'src_bytes', 'dst_bytes', 'conn_state', 'missed_bytes', 'src_pkts', 'src_ip_bytes', 'dst_pkts', 'dst_ip_bytes', 'dns_query', 'dns_qclass', 'dns_qtype', 'dns_rcode', 'dns_AA', 'dns_RD', 'dns_RA', 'dns_rejected', 'ssl_version', 'ssl_cipher', 'ssl_resumed', 'ssl_established', 'ssl_subject', 'ssl_issuer', 'http_trans_depth', 'http_method', 'http_uri', 'http_version', 'http_request_body_len', 'http_response_body_len', 'http_status_code', 'http_user_agent', 'http_orig_mime_types', 'http_resp_mime_types', 'weird_name', 'weird_addl', 'weird_notice', 'label', 'type']

Label counts :
label
1    161043
0     50000
Name: count, dtype: int64

Type counts  :
type
normal        50000
backdoor      20000
ddos          20000
dos           20000
injection     20000
password      20000
ransomware    20000
scanning      20000
xss           20000
mitm           1043
Name: count, dtype: in

## Stage 2 — Data Cleaning & Label Harmonisation

In [2]:
import numpy as np

FAMILY_MAP = {
    'normal':    'Benign',
    'backdoor':  'Malware',
    'ddos':      'DDoS',
    'dos':       'DoS',
    'injection': 'Web',
    'mitm':      'Spoofing',
    'password':  'BruteForce',
    'ransomware':'Malware',
    'scanning':  'Recon',
    'xss':       'Web',
}

UNIFIED_MAP = {
    'normal':    'BenignTraffic',
    'backdoor':  'Malware-Backdoor',
    'ddos':      'DDoS-TCPFlood',
    'dos':       'DoS-HTTPFlood',
    'injection': 'Web-SQLi',
    'mitm':      'Spoofing-ARP',
    'password':  'BruteForce-Dict',
    'ransomware':'Malware-Backdoor',
    'scanning':  'Recon-PortScan',
    'xss':       'Web-SQLi',
}

df = df_ton.copy()
df['type_clean'] = df['type'].str.strip().str.lower().fillna('normal')

df['attack_family'] = df['type_clean'].map(FAMILY_MAP).fillna('Unknown')
df['unified_label']  = df['type_clean'].map(UNIFIED_MAP).fillna('BenignTraffic')
df['is_attack']      = df['label'].astype(int) == 1   # ← FIXED: integer comparison

df_harmonized = pd.DataFrame({
    'dataset_source'   : 'TON_IoT',
    'source_file'      : os.path.basename(TON_PATH),
    'sample_seed'      : None,
    'phase'            : 2,
    'protocol'         : df['proto'].str.upper().str.strip(),
    'duration'         : pd.to_numeric(df['duration'],      errors='coerce'),
    'src_bytes'        : pd.to_numeric(df['src_bytes'],     errors='coerce'),
    'dst_bytes'        : pd.to_numeric(df['dst_bytes'],     errors='coerce'),
    'total_bytes'      : pd.to_numeric(df['src_bytes'],     errors='coerce') +
                         pd.to_numeric(df['dst_bytes'],     errors='coerce'),
    'src_pkts'         : pd.to_numeric(df['src_pkts'],      errors='coerce'),
    'dst_pkts'         : pd.to_numeric(df['dst_pkts'],      errors='coerce'),
    'total_pkts'       : pd.to_numeric(df['src_pkts'],      errors='coerce') +
                         pd.to_numeric(df['dst_pkts'],      errors='coerce'),
    'flow_rate_byts_s' : None,
    'flow_rate_pkts_s' : None,
    'syn_flag'         : None,
    'ack_flag'         : None,
    'fin_flag'         : None,
    'rst_flag'         : None,
    'cic_iat_mean'     : None,
    'cic_iat_std'      : None,
    'cic_flow_pkts_s'  : None,
    'cic_active_mean'  : None,
    'cic_idle_mean'    : None,
    'ton_conn_state'   : df['conn_state'].astype(str).str.strip(),
    'ton_dns_query'    : df['dns_query'].astype(str).str.strip(),
    'ton_http_method'  : df['http_method'].astype(str).str.strip(),
    'label_raw'        : df['label'].astype(str).str.strip(),
    'unified_label'    : df['unified_label'],
    'attack_family'    : df['attack_family'],
    'is_attack'        : df['is_attack'],
    'ingest_timestamp' : pd.Timestamp.now(),
})

df_harmonized.replace({'-': None, 'nan': None, '': None}, inplace=True)

print(f"Harmonized shape     : {df_harmonized.shape}")
print(f"is_attack True/False : {df_harmonized['is_attack'].value_counts().to_dict()}")
print(f"\nunified_label breakdown:\n{df_harmonized['unified_label'].value_counts()}")

Harmonized shape     : (211043, 31)
is_attack True/False : {True: 161043, False: 50000}

unified_label breakdown:
unified_label
BenignTraffic       50000
Malware-Backdoor    40000
Web-SQLi            40000
DDoS-TCPFlood       20000
DoS-HTTPFlood       20000
BruteForce-Dict     20000
Recon-PortScan      20000
Spoofing-ARP         1043
Name: count, dtype: int64


## Stage 3 — PostgreSQL Bulk Ingestion

In [3]:
import sys, os
sys.path.insert(0, os.path.expanduser("~/sentinelmesh"))
from etl.utils.db_connect import get_engine
engine = get_engine()

CHUNK = 5000
total = len(df_harmonized)
loaded = 0
for i in range(0, total, CHUNK):
    chunk = df_harmonized.iloc[i:i+CHUNK]
    chunk.to_sql('stg_harmonized', engine, if_exists='append', index=False, method='multi')
    loaded += len(chunk)
    print(f"Loaded {loaded}/{total} rows...", end="\r")
print(f"✅ TON_IoT ETL complete: {loaded} rows inserted into stg_harmonized")

✅ TON_IoT ETL complete: 211043 rows inserted into stg_harmonized


## Stage 4 — Verification & Re-run Guard

In [4]:
from sqlalchemy import text
import pandas as pd
import os

# ── Step 1: Wipe existing TON_IoT rows ───────────────────────────────────────
with engine.connect() as con:
    con.execute(text("DELETE FROM stg_harmonized WHERE dataset_source = 'TON_IoT'"))
    con.commit()
    result = con.execute(text("SELECT COUNT(*) FROM stg_harmonized WHERE dataset_source = 'TON_IoT'"))
    print(f"TON_IoT rows after wipe: {result.scalar():,}")  # should be 0

# ── Step 2: Reload from CSV (single run) ─────────────────────────────────────
TON_PATH = os.path.expanduser("~/sentinelmesh/data/TON_IoT/Train_Test_datasets/Train_Test_Network_dataset/train_test_network.csv")
df_ton = pd.read_csv(TON_PATH, low_memory=False)

FAMILY_MAP = {
    'normal':'Benign','backdoor':'Malware','ddos':'DDoS','dos':'DoS',
    'injection':'Web','mitm':'Spoofing','password':'BruteForce',
    'ransomware':'Malware','scanning':'Recon','xss':'Web',
}
UNIFIED_MAP = {
    'normal':'BenignTraffic','backdoor':'Malware-Backdoor','ddos':'DDoS-TCPFlood',
    'dos':'DoS-HTTPFlood','injection':'Web-SQLi','mitm':'Spoofing-ARP',
    'password':'BruteForce-Dict','ransomware':'Malware-Backdoor',
    'scanning':'Recon-PortScan','xss':'Web-SQLi',
}

df = df_ton.copy()
df['type_clean']     = df['type'].str.strip().str.lower().fillna('normal')
df['attack_family']  = df['type_clean'].map(FAMILY_MAP).fillna('Unknown')
df['unified_label']  = df['type_clean'].map(UNIFIED_MAP).fillna('BenignTraffic')
df['is_attack']      = df['label'].astype(int) == 1

df_harmonized = pd.DataFrame({
    'dataset_source'   : 'TON_IoT',
    'source_file'      : os.path.basename(TON_PATH),
    'sample_seed'      : None,
    'phase'            : 2,
    'protocol'         : df['proto'].str.upper().str.strip(),
    'duration'         : pd.to_numeric(df['duration'],    errors='coerce'),
    'src_bytes'        : pd.to_numeric(df['src_bytes'],   errors='coerce'),
    'dst_bytes'        : pd.to_numeric(df['dst_bytes'],   errors='coerce'),
    'total_bytes'      : pd.to_numeric(df['src_bytes'],   errors='coerce') +
                         pd.to_numeric(df['dst_bytes'],   errors='coerce'),
    'src_pkts'         : pd.to_numeric(df['src_pkts'],    errors='coerce'),
    'dst_pkts'         : pd.to_numeric(df['dst_pkts'],    errors='coerce'),
    'total_pkts'       : pd.to_numeric(df['src_pkts'],    errors='coerce') +
                         pd.to_numeric(df['dst_pkts'],    errors='coerce'),
    'flow_rate_byts_s' : None, 'flow_rate_pkts_s' : None,
    'syn_flag'         : None, 'ack_flag'          : None,
    'fin_flag'         : None, 'rst_flag'           : None,
    'cic_iat_mean'     : None, 'cic_iat_std'        : None,
    'cic_flow_pkts_s'  : None, 'cic_active_mean'    : None,
    'cic_idle_mean'    : None,
    'ton_conn_state'   : df['conn_state'].astype(str).str.strip(),
    'ton_dns_query'    : df['dns_query'].astype(str).str.strip(),
    'ton_http_method'  : df['http_method'].astype(str).str.strip(),
    'label_raw'        : df['label'].astype(str).str.strip(),
    'unified_label'    : df['unified_label'],
    'attack_family'    : df['attack_family'],
    'is_attack'        : df['is_attack'],
    'ingest_timestamp' : pd.Timestamp.now(),
})
df_harmonized.replace({'-': None, 'nan': None, '': None}, inplace=True)

# ── Step 3: Single load, no re-runs ──────────────────────────────────────────
CHUNK = 5000
total = len(df_harmonized)
loaded = 0

for i in range(0, total, CHUNK):
    chunk = df_harmonized.iloc[i:i+CHUNK]
    chunk.to_sql('stg_harmonized', engine, if_exists='append',
                 index=False, method='multi')
    loaded += len(chunk)
    print(f"  Loaded {loaded:,} / {total:,} rows...", end='\r')

print(f"\n✅ Reload complete — {loaded:,} rows inserted")

# ── Step 4: Final verification ────────────────────────────────────────────────
with engine.connect() as con:
    result = con.execute(text("""
        SELECT dataset_source, COUNT(*) AS total_rows,
               SUM(CASE WHEN is_attack THEN 1 ELSE 0 END) AS attacks,
               SUM(CASE WHEN NOT is_attack THEN 1 ELSE 0 END) AS benign
        FROM stg_harmonized
        GROUP BY dataset_source ORDER BY dataset_source
    """))
    df_v = pd.DataFrame(result.fetchall(), columns=result.keys())

print(df_v.to_string(index=False))
print(f"\nGrand total: {df_v['total_rows'].sum():,}")

TON_IoT rows after wipe: 0
  Loaded 211,043 / 211,043 rows...
✅ Reload complete — 211,043 rows inserted
dataset_source  total_rows  attacks  benign
    CICIoT2023      241527   221527   20000
       TON_IoT      211043   161043   50000

Grand total: 452,570


In [5]:
import pandas as pd
import numpy as np
import datetime
from sqlalchemy import text

# ── Pull from stg_harmonized ──────────────────────────────────────────────────
with engine.connect() as con:
    df = pd.read_sql("""
        SELECT dataset_source, unified_label, attack_family,
               is_attack, protocol, duration, total_bytes, total_pkts
        FROM stg_harmonized
    """, con)

print(f"Loaded {len(df):,} rows  |  attacks: {df['is_attack'].sum():,}  |  benign: {(~df['is_attack']).sum():,}")

# ── Assign device_id & risk_date deterministically ───────────────────────────
N_DEVICES  = 50
N_DAYS     = 90
end_date   = datetime.date(2026, 6, 11)
date_range = [(end_date - datetime.timedelta(days=N_DAYS - 1 - i)) for i in range(N_DAYS)]

# Assign round-robin so every device gets every date
df = df.reset_index(drop=True)
df['device_id'] = ['DEV_' + str((i % N_DEVICES) + 1).zfill(3) for i in range(len(df))]
df['risk_date'] = [date_range[(i // N_DEVICES) % N_DAYS]        for i in range(len(df))]

print(f"Unique devices: {df['device_id'].nunique()}  |  Unique dates: {df['risk_date'].nunique()}")

# ── Aggregate ─────────────────────────────────────────────────────────────────
grp = df.groupby(['device_id', 'risk_date', 'dataset_source'])

agg = grp.agg(
    total_flows  = ('is_attack', 'count'),
    attack_flows = ('is_attack', 'sum'),
    total_bytes  = ('total_bytes', 'sum'),
    total_pkts   = ('total_pkts', 'sum'),
    avg_duration = ('duration', 'mean'),
).reset_index()

agg['benign_flows'] = agg['total_flows'] - agg['attack_flows']
agg['attack_ratio'] = agg['attack_flows'] / agg['total_flows']

# Top attack family per group
top_family = (
    df[df['is_attack']]
    .groupby(['device_id', 'risk_date', 'dataset_source'])['attack_family']
    .agg(lambda x: x.value_counts().index[0])
    .reset_index()
    .rename(columns={'attack_family': 'top_attack_family'})
)
agg = agg.merge(top_family, on=['device_id', 'risk_date', 'dataset_source'], how='left')
agg['top_attack_family'] = agg['top_attack_family'].fillna('None')

# Unique attack types per group
unique_types = (
    df[df['is_attack']]
    .groupby(['device_id', 'risk_date', 'dataset_source'])['unified_label']
    .nunique()
    .reset_index()
    .rename(columns={'unified_label': 'unique_attack_types'})
)
agg = agg.merge(unique_types, on=['device_id', 'risk_date', 'dataset_source'], how='left')
agg['unique_attack_types'] = agg['unique_attack_types'].fillna(0).astype(int)

# Dominant protocol per group
dom_proto = (
    df.groupby(['device_id', 'risk_date', 'dataset_source'])['protocol']
    .agg(lambda x: x.value_counts().index[0])
    .reset_index()
    .rename(columns={'protocol': 'dominant_protocol'})
)
agg = agg.merge(dom_proto, on=['device_id', 'risk_date', 'dataset_source'], how='left')

# ── Risk score ────────────────────────────────────────────────────────────────
agg['risk_score'] = (
    0.60 * agg['attack_ratio'] +
    0.25 * (agg['unique_attack_types'] / 10).clip(0, 1) +
    0.15 * agg['attack_ratio']
).clip(0, 1) * 100

agg['risk_tier'] = pd.cut(
    agg['risk_score'],
    bins=[-0.1, 25, 50, 75, 100],
    labels=['Low', 'Medium', 'High', 'Critical']
).astype(str)

print(f"\nAggregated into {len(agg):,} device-day records")
print(agg['risk_tier'].value_counts().to_string())
print(f"\nSample:\n{agg[['device_id','risk_date','attack_ratio','risk_score','risk_tier']].head(10).to_string(index=False)}")

Loaded 452,570 rows  |  attacks: 382,570  |  benign: 70,000
Unique devices: 50  |  Unique dates: 90

Aggregated into 9,000 device-day records
risk_tier
Critical    4500
High        4500

Sample:
device_id  risk_date  attack_ratio  risk_score risk_tier
  DEV_001 2026-03-14      0.907407   93.055556  Critical
  DEV_001 2026-03-14      0.765957   72.446809      High
  DEV_001 2026-03-15      0.907407   93.055556  Critical
  DEV_001 2026-03-15      0.765957   72.446809      High
  DEV_001 2026-03-16      0.907407   93.055556  Critical
  DEV_001 2026-03-16      0.765957   72.446809      High
  DEV_001 2026-03-17      0.907407   93.055556  Critical
  DEV_001 2026-03-17      0.765957   72.446809      High
  DEV_001 2026-03-18      0.907407   93.055556  Critical
  DEV_001 2026-03-18      0.765957   72.446809      High


In [6]:
from sqlalchemy import text

# Clear any previous run
with engine.connect() as con:
    con.execute(text("TRUNCATE TABLE fact_device_risk_daily RESTART IDENTITY"))
    con.commit()

# Write
agg.to_sql('fact_device_risk_daily', engine,
           if_exists='append', index=False, method='multi', chunksize=5000)

# Verify
with engine.connect() as con:
    result = con.execute(text("""
        SELECT risk_tier, COUNT(*) AS records, ROUND(AVG(risk_score)::numeric, 2) AS avg_score
        FROM fact_device_risk_daily
        GROUP BY risk_tier ORDER BY avg_score DESC
    """))
    df_v = pd.DataFrame(result.fetchall(), columns=result.keys())

print(df_v.to_string(index=False))
print(f"\n✅ fact_device_risk_daily populated — {len(agg):,} records written")

risk_tier  records avg_score
 Critical     4500     93.79
     High     4500     72.81

✅ fact_device_risk_daily populated — 9,000 records written


In [7]:
import pandas as pd
import datetime
from sqlalchemy import text

# Pull attacks only
with engine.connect() as con:
    df_atk = pd.read_sql("""
        SELECT dataset_source, attack_family, unified_label,
               protocol, total_bytes, total_pkts, duration
        FROM stg_harmonized
        WHERE is_attack = TRUE
    """, con)

print(f"Loaded {len(df_atk):,} attack rows")

# Assign event_date using same deterministic approach as fact_device_risk_daily
N_DAYS     = 90
end_date   = datetime.date(2026, 6, 11)
date_range = [(end_date - datetime.timedelta(days=N_DAYS - 1 - i)) for i in range(N_DAYS)]

df_atk = df_atk.reset_index(drop=True)
df_atk['event_date'] = [date_range[i % N_DAYS] for i in range(len(df_atk))]

# Aggregate by date + source + attack family + unified label + protocol
agg_atk = df_atk.groupby(
    ['event_date', 'dataset_source', 'attack_family', 'unified_label', 'protocol']
).agg(
    flow_count   = ('unified_label', 'count'),
    total_bytes  = ('total_bytes',   'sum'),
    total_pkts   = ('total_pkts',    'sum'),
    avg_duration = ('duration',      'mean'),
).reset_index()

print(f"Aggregated into {len(agg_atk):,} timeline records")
print(f"Date range: {agg_atk['event_date'].min()} → {agg_atk['event_date'].max()}")
print(f"Attack families: {agg_atk['attack_family'].nunique()}")
print(f"Unique labels: {agg_atk['unified_label'].nunique()}")

# Write to Postgres
with engine.connect() as con:
    con.execute(text("TRUNCATE TABLE fact_attack_timeline RESTART IDENTITY"))
    con.commit()

agg_atk.to_sql('fact_attack_timeline', engine,
               if_exists='append', index=False, method='multi', chunksize=5000)

# Verify
with engine.connect() as con:
    result = con.execute(text("""
        SELECT attack_family, COUNT(*) AS records, SUM(flow_count) AS total_flows
        FROM fact_attack_timeline
        GROUP BY attack_family ORDER BY total_flows DESC
    """))
    df_v = pd.DataFrame(result.fetchall(), columns=result.keys())

print(df_v.to_string(index=False))
print(f"\n✅ fact_attack_timeline populated — {len(agg_atk):,} records written")

Loaded 382,570 attack rows
Aggregated into 3,806 timeline records
Date range: 2026-03-14 → 2026-06-11
Attack families: 8
Unique labels: 15
attack_family  records  total_flows
         DDoS      496        80000
        Recon      698        60000
          DoS      375        60000
          Web      396        45245
      Malware      461        43218
     Spoofing      756        41043
   BruteForce      438        33064
        Mirai      186        20000

✅ fact_attack_timeline populated — 3,806 records written


## Stage 5 — ETL Sign-off & Pipeline Summary

**ETL status:** TON_IoT data successfully harmonised and loaded into `stg_harmonized`.

| Step | Output |
|---|---|
| Raw records loaded | 211,043 rows × 44 columns |
| Harmonised shape | 211,043 rows × 31 columns |
| Rows in `stg_harmonized` (TON_IoT) | 211,043 |
| `fact_attack_timeline` records | 3,806 |
| Date range | 2026-03-14 → 2026-06-11 |
| Attack families | 8 |

**Label taxonomy applied:**

| Raw type | `attack_family` | `unified_label` |
|---|---|---|
| `normal` | Benign | BenignTraffic |
| `backdoor` | Malware | Malware-Backdoor |
| `ddos` | DDoS | DDoS-TCPFlood |
| `injection` | Web | Web-SQLi |
| `mitm` | Spoofing | Spoofing-ARP |

> **Idempotency:** Stage 4 wipes existing TON_IoT rows before re-inserting, making the full pipeline safely re-runnable.

**Next step:** Run `04_prediction_writeback.ipynb` to execute batch inference over the combined `stg_harmonized` table.